## Issues

Issue is a record describing a problem or other phenomena localized in a specific area of the scene.
Issue is represented as a set of labels, typically including:

1. Location:
   1. dataset
   2. scene
   3. additional labels identifying scene (together referred below as Scene)
   4. area (polygon)
2. Categorization:
   1. category (unique code of the issue type)
   2. description of the category (cat_description)
   3. visualization hints of the category (optional, each with its own label)
3. Issue Specific Descriptions (optional):
   1. description - textual description
   2. numerical descriptors (each with its own label) - e.g. number of pixel in the area.
   3. optional visualization hints  (each with its own label)


### Code Structure

Collection of issues is managed by class IssueCollection, which is maintaining a DataFrame catalog of issues as rows, 
and allowing to add, remove, search and show issues.

In particular:
```issues.qix()``` query create another instance of Issues with requested subset of the issues.

Visualization of issues is created by class VisIssue.

```vis_issue(issues, fig, axes_queries: list[Axes] | Axes | list[tuple[Axes, SceneLabels]])```

1. vis_issue.show() - displays issues from scenes associated with given axes.
2. The association may be either explicit when axes are list of tuples, or implicit if an axis carries a particular attribute  ```axis._scene_labels: SceneLabels```
3. ```Issues.SCENE_LABELS='_scene_labels'```
4. ```Issues.attach_label(obj: object | list[objects], labels)-> object | list[objects]```
   1. Helper class method to attach SCENE_LABELS attribute to given object(s)


### Use Cases:

Below we assume that:

1. scene includes all the identifiers, like dataset, scene, etc

2. there are multiple visualizers producing  matplotlib.pyplot.Figure 

In [7]:
%matplotlib widget
import matplotlib as mpl

mpl.rcParams['figure.dpi'] = 80
import warnings
warnings.filterwarnings('ignore')
from inu.visual.insight import imgrid
from inu.visual.annotations import IssueCollection, VisIssue, show_issues_on_scenes
from inu.utils.label import Labels
from inu.pipelines.benchmark.core import Benchmark
from algutils.env import EnvLoc  # noqa: F401
import logging
for _n in ('resman', 'engine', 'bmk', 'datacast'):
    logging.getLogger(_n).setLevel(logging.DEBUG)
logging.getLogger('engine').setLevel(logging.INFO)
from inu.datacast import resman
resman.get("col").discover("/sp/code/algodev/resources/collections/", cache=False)

(!) ---> Log file: /tmp/logs/algodev.log


1

In [6]:

bm = Benchmark('enhance_gm_v4_selector')
bm.exec_pipeline(measure=False)
alg_name = 'MoFcV4_4_UC_select_3_class'

ResNotFoundError: No Collection with name='DISP_EVAL_SYNTH' among available = 'KITTI'

In [ ]:
dc = bm.evaluated_ds().qix(scene=['A_0000_0007', 'B_0047_0012', 'C_0034_0010', 'C_0095_0011', 'A_0000_0009'], as_dc=True) # DataCollection
issues = IssueCollection.from_csv('~/code/algodev/resources/issues/selector_issues.csv')

In [ ]:
from inu.pipelines.adapters.disparity.visual import comp_sel_warp_missed_bad_dpe

vis = lambda _ : comp_sel_warp_missed_bad_dpe(_, alg_name, out='all')
# vis = lambda _ : imgrid(_.GT.L, out='all')

In [ ]:
# import matplotlib.pyplot as plt
# fig = plt.figure()
# axs = fig.subplots
# fig, axes = plt.subplots()
# fig.canvas.supports_blit

#### Use Case 1 - Show one scene of one alg (one or more axes), then annotate

In [ ]:
scene_lbls = Labels(dataset='FT3D', scene='A_0000_0007', )

axes_queries = [(0, scene_lbls), (2, scene_lbls)]
vis_outs = vis(dc.qix(**scene_lbls, trans=True))
# vis_issue = VisIssue(issues, vis_outs.fig, axes_queries, scene_lbls | dict(alg=alg_name))
vis_issue = VisIssue(issues, vis_outs.fig, axes_queries, scene_lbls )

vis_issue.show()

#### Use Case 2: Show one scene (one or more axes) for two alg

In [ ]:
# ToDO In Imgrid - Include all labels of image inside the ax
scene_lbls = Labels(dataset='FT3D', scene='A_0000_0007', )
axes_queries = [(0, scene_lbls | dict(alg=alg_name)), (1, scene_lbls | dict(alg='TEST'))]

vis_two_ims = lambda _ : imgrid(_.GT.L, _.GT.L,  out='all')
vis_outs = vis_two_ims(dc.qix(**scene_lbls, trans=True))

vis_issue = VisIssue(issues, vis_outs.fig, axes_queries)
vis_issue.show()

#### Use Case 3: Show specific issues on their **relevant** scenes

In [ ]:
specific_issues = issues.qix(issue_type=['S2', 'S4'])
show_issues_on_scenes(specific_issues, dc=dc, vis=vis, axes_queries=[3])

#### Use Case 4: After showing all scenes, Show specific issues on them ( for one alg)

In [ ]:
for scene_labels, scene_imgs in dc.iter(group=['dataset', 'scene']):
    scene_imgs = scene_imgs.fetch()
    vis_outs = vis(scene_imgs)
    VisIssue.attach_labels(vis_outs.fig.axes, labels=scene_labels)
    vis_issue = VisIssue(issues.qix(alg_name, issue_type=['S2', 'S4']), vis_outs.fig)
    vis_issue.show()